# Solar Potential Estimation


In [ ]:
import os
import numpy as np
import rasterio
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, Model

Calculate Solar Potential

In [ ]:
import os
import rasterio
from rasterio.features import shapes
import geopandas as gpd
import pvlib
import pandas as pd
from shapely.geometry import shape
import numpy as np
from PIL import Image
from pyproj import Transformer
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras import backend as K
from tensorflow.keras.saving import register_keras_serializable
from tensorflow.keras.losses import BinaryCrossentropy
import tempfile

# =======================
# Paths
# =======================
INPUT_TIF_PATH = "/content/drive/MyDrive/Solar_Potential_Project/data/images/sat_1.tif"
MODEL_PATH = "/content/drive/MyDrive/Solar_Potential_Project/final.keras"  # Updated to keras format

# =======================
# Custom Metrics
# =======================
@register_keras_serializable()
def iou_metric(y_true, y_pred, smooth=1):
    # Cast inputs to float32 to ensure compatible types for operations
    y_true = K.cast(y_true, dtype='float32')
    y_pred = K.cast(y_pred, dtype='float32')
    intersection = K.sum(K.abs(y_true * y_pred), axis=[1, 2, 3])
    union = K.sum(y_true, [1, 2, 3]) + K.sum(y_pred, [1, 2, 3]) - intersection
    return K.mean((intersection + smooth) / (union + smooth), axis=0)

@register_keras_serializable()
def dice_coefficient(y_true, y_pred, smooth=1):
    # Cast inputs to float32 to ensure compatible types for operations
    y_true = K.cast(y_true, dtype='float32')
    y_pred = K.cast(y_pred, dtype='float32')
    intersection = K.sum(y_true * y_pred, axis=[1, 2, 3])
    union = K.sum(y_true, axis=[1, 2, 3]) + K.sum(y_pred, axis=[1, 2, 3])
    return K.mean((2. * intersection + smooth) / (union + smooth), axis=0)

@register_keras_serializable()
def threshold_accuracy(y_true, y_pred):
    # Cast inputs to float32 to ensure compatible types for operations
    y_true = K.cast(y_true, dtype='float32')
    y_pred = K.cast(y_pred, dtype='float32')
    return K.mean(K.equal(y_true, K.cast(y_pred > 0.5, y_true.dtype)))

@register_keras_serializable()
def bce_dice_loss(y_true, y_pred):
    # Cast inputs to float32 to ensure compatible types for operations
    y_true = K.cast(y_true, dtype='float32')
    y_pred = K.cast(y_pred, dtype='float32')
    bce = BinaryCrossentropy()(y_true, y_pred)
    intersection = K.sum(y_true * y_pred)
    dice = (2. * intersection + 1) / (K.sum(y_true) + K.sum(y_pred) + 1)
    return bce + (1 - dice)

@register_keras_serializable()
def f1_score(y_true, y_pred, smooth=1):
    # Cast inputs to float32 to ensure compatible types for operations
    y_true = K.cast(y_true, dtype='float32')
    y_pred = K.cast(y_pred, dtype='float32')
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    y_pred_f = K.cast(y_pred_f > 0.5, dtype='float32')
    tp = K.sum(y_true_f * y_pred_f)
    precision = tp / (K.sum(y_pred_f) + K.epsilon())
    recall = tp / (K.sum(y_true_f) + K.epsilon())
    return 2 * (precision * recall) / (precision + recall + K.epsilon())

# =======================
# Load Model
# =======================
custom_objects = {
    'iou_metric': iou_metric,
    'dice_coefficient': dice_coefficient,
    'threshold_accuracy': threshold_accuracy,
    'bce_dice_loss': bce_dice_loss,
    'f1_score': f1_score
}
model = load_model(MODEL_PATH, custom_objects=custom_objects)
input_size = model.input_shape[1:3]

# =======================
# Parameters
# =======================
efficiency = 18  # %
tilt_angle = 30  # degrees
azimuth = 180    # degrees from North

# =======================
# Processing Function
# =======================
def process_image(path):
    with rasterio.open(path) as src:
        img = src.read()
        transform = src.transform
        crs = src.crs
        bounds = src.bounds

        transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
        center_lon, center_lat = transformer.transform(
            (bounds.left + bounds.right) / 2,
            (bounds.bottom + bounds.top) / 2
        )

        if img.shape[0] > 3:
            img = img[:3]
        img = np.transpose(img, (1, 2, 0)) / 255.0

        resized_img = np.array(Image.fromarray((img * 255).astype(np.uint8)).resize(input_size)) / 255.0
        pred_prob = model.predict(np.expand_dims(resized_img, axis=0))[0, ..., 0]
        pred_mask = (pred_prob > 0.5).astype(np.uint8)

        # Save temporary mask as GeoTIFF
        with tempfile.NamedTemporaryFile(delete=False, suffix=".tif") as tmp_mask:
            with rasterio.open(
                tmp_mask.name,
                "w",
                driver="GTiff",
                height=pred_mask.shape[0],
                width=pred_mask.shape[1],
                count=1,
                dtype=pred_mask.dtype,
                crs=crs,
                transform=transform
            ) as dst:
                dst.write(pred_mask, 1)

        # Convert to polygons
        with rasterio.open(tmp_mask.name) as mask_src:
            gdf = gpd.GeoDataFrame.from_features(
                [{"geometry": shape(s), "properties": {"value": v}}
                 for s, v in shapes(mask_src.read(1), transform=transform) if v == 1],
                crs=crs
            )

        if gdf.empty:
            print("No rooftops detected.")
            return None

        # Area Calculation
        utm_zone = int((center_lon + 180) / 6) + 1
        epsg_code = f"EPSG:{'326' if center_lat >= 0 else '327'}{utm_zone}"
        gdf_utm = gdf.to_crs(epsg_code)
        gdf["area_m2"] = gdf_utm.geometry.area

        # Solar Calculations
        location = pvlib.location.Location(center_lat, center_lon, tz="UTC")
        times = pd.date_range("2023-01-01", "2023-12-31", freq="h", tz="UTC")
        solpos = location.get_solarposition(times)
        clearsky = location.get_clearsky(times)

        irradiance = pvlib.irradiance.get_total_irradiance(
            surface_tilt=tilt_angle,
            surface_azimuth=azimuth,
            solar_zenith=solpos["apparent_zenith"],
            solar_azimuth=solpos["azimuth"],
            dni=clearsky["dni"],
            ghi=clearsky["ghi"],
            dhi=clearsky["dhi"]
        )

        annual_irrad = irradiance["poa_global"].sum() / 1000  # kWh/m^2/year
        gdf["energy_kWh"] = gdf["area_m2"] * annual_irrad * (efficiency / 100)

        # Print Calculations
        print("\n===== Rooftop Solar Potential Estimation =====")
        print(f"Image Center Location: Latitude = {center_lat:.6f}, Longitude = {center_lon:.6f}")
        print(f"Total Detected Rooftop Area: {gdf['area_m2'].sum():.2f} m²")
        print(f"Annual POA Irradiance: {annual_irrad:.2f} kWh/m²/year")
        print(f"Estimated Total Annual Energy Output: {gdf['energy_kWh'].sum():.2f} kWh/year")
        print("\n")

        return gdf, center_lat, center_lon, annual_irrad

# =======================
# Run
# =======================
results = process_image(INPUT_TIF_PATH)
if results:
    gdf, lat, lon, irrad = results

    # Load the original satellite image (RGB) for display
    with rasterio.open(INPUT_TIF_PATH) as src:
        img = src.read([1, 2, 3])  # RGB channels only
        img = np.transpose(img, (1, 2, 0))  # CHW -> HWC
        img = img / img.max()  # Normalize to [0, 1] for display

    # Create side-by-side figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

    ax1.imshow(img)
    ax1.set_title("Original Satellite Image")
    ax1.set_xlabel("Pixel X")
    ax1.set_ylabel("Pixel Y")
    ax1.grid(False)

    gdf.plot(column="energy_kWh", ax=ax2, legend=True,
             cmap="YlOrRd", legend_kwds={'label': "Estimated kWh/year"})
    ax2.set_title(f"Solar Potential Map\nLat: {lat:.4f}°, Lon: {lon:.4f}°")
    ax2.set_xlabel("Longitude")
    ax2.set_ylabel("Latitude")
    ax2.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()